## Create tables for describing data and their variables

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

EXPERIMENTS_ROOT = Path("/data/shared/fsibilla/clean_code/Q1/experiments")

EXPERIMENT_LABELS = {
    "eth_micron": "Eth ESS",
    "lka_micron": "Lka HIES",
    "lka_vam":    "Lka VAM",
    "moz_vam":    "Moz VAM",
    "nga_micron": "Nga NLSS",
    "nga_mics":   "Nga MICS",
    "yem_mvam":   "Yem mVAM",
    "zwe_mics":   "Zwe MICS",
}

VARIABLE_LABELS = {
    "avg_adult_education": "Education",
    "fe_ai":               "Iron",
    "fol_ai":              "Folate",
    "log_exp":             "Expenditures",
    "va_ai":               "Vitamin A",
    "vb12_ai":             "Vitamin B12",
    "zn_ai":               "Zinc",
    "fe_mg":               "Iron",
    "folate_mcg":          "Folate",
    "vita_rae_mcg":        "Vitamin A",
    "vitb12_mcg":          "Vitamin B12",
    "zn_mg":               "Zinc",
    "FCS":                 "FCS",
    "FES":                 "FES",
    "education_score":     "Education",
    "log_income":          "Income",
    "rCSI":                "rCSI",
    "space_per_person":    "Space pp",
    "FGHIron":             "Iron",
    "FGProtein":           "Protein",
    "FGVitA":              "Vitamin A",
    "wscore":              "Wealth Score",
    "log_exp_pp":          "Expenditures pp",
}

# adm1 column per experiment
ADM1_COL = {
    "eth_micron": "adm1name",
    "lka_micron": "adm1name",
    "lka_vam":    "adm1name",
    "moz_vam":    "adm1name",
    "nga_micron": "adm1name",
    "nga_mics":   "adm1name",
    "yem_mvam":   "adm1name",
    "zwe_mics":   "adm1name",
}

# target variables per experiment (from figure)
EXPERIMENT_VARS = {
    "eth_micron": ["avg_adult_education", "fe_ai",  "fol_ai",      "log_exp", "va_ai",        "vb12_ai",    "zn_ai"],
    "lka_micron": ["avg_adult_education", "fe_mg",  "folate_mcg",  "log_exp", "vita_rae_mcg", "vitb12_mcg", "zn_mg"],
    "lka_vam":    ["FCS", "FES", "education_score", "log_income", "rCSI", "space_per_person"],
    "moz_vam":    ["FCS", "FGHIron", "FGProtein", "FGVitA", "rCSI"],
    "nga_micron": ["avg_adult_education", "fe_ai",  "fol_ai",      "log_exp", "va_ai",        "vb12_ai",    "zn_ai"],
    "nga_mics":   ["avg_adult_education", "space_per_person", "wscore"],
    "yem_mvam":   ["FCS", "log_exp_pp", "rCSI"],
    "zwe_mics":   ["avg_adult_education", "space_per_person", "wscore"],
}

In [5]:
rows = []

for exp_id, exp_label in EXPERIMENT_LABELS.items():
    path = EXPERIMENTS_ROOT / exp_id / "full.csv"
    if not path.exists():
        print(f"Missing: {path}")
        continue

    df = pd.read_csv(path).replace([np.inf, -np.inf], np.nan)
    adm1_col = ADM1_COL[exp_id]
    n_points = len(df)
    n_adm1   = df[adm1_col].nunique() if adm1_col in df.columns else np.nan

    for raw_var in EXPERIMENT_VARS[exp_id]:
        if raw_var not in df.columns:
            print(f"  [{exp_id}] missing column: {raw_var}")
            continue
        s = pd.to_numeric(df[raw_var], errors="coerce").replace([np.inf, -np.inf], np.nan)
        rows.append({
            "Dataset":  exp_label,
            "N points": n_points,
            "N adm1":   n_adm1,
            "Variable": VARIABLE_LABELS.get(raw_var, raw_var),
            "Min":      round(s.min(), 3),
            "Median":   round(s.median(), 3),
            "Max":      round(s.max(), 3),
        })

summary_df = pd.DataFrame(rows)
summary_df = summary_df.set_index(["Dataset", "N points", "N adm1", "Variable"])
summary_df

Min   Median       Max
Dataset  N points N adm1 Variable                                  
Eth ESS  6214     11     Education         0.000    3.500    35.000
                         Iron              0.000   26.550   370.208
                         Folate            0.000  216.631  3722.102
                         Expenditures      0.000    5.549    10.964
                         Vitamin A         0.000  189.443  7250.927
                         Vitamin B12       0.000    0.209    12.731
                         Zinc              0.000   12.997   110.379
Lka HIES 19911    9      Education         0.000   10.000    19.500
                         Iron              0.007   12.768    56.004
                         Folate            0.041  202.338   756.106
                         Expenditures      5.746   12.080    18.436
                         Vitamin A         0.000  199.166  1324.646
                         Vitamin B12       0.000    1.114    11.105
                         Zinc              0.002    9.016    28.879
Lka VAM  14490    9      FCS              12.500   49.000   112.000
                         FES               0.007    0.554     0.960
                         Education         0.000    0.565     1.000
                         Income            9.210   11.408    14.914
                         rCSI              0.000    2.000    56.000
                         Space pp          0.000    0.750     6.000
Moz VAM  11080    11     FCS               0.000   39.000   112.000
                         Iron              0.000    3.000    21.000
                         Protein           0.000    4.000    42.000
                         Vitamin A         0.000    5.000    42.000
                         rCSI              0.000    3.000    56.000
Nga NLSS 22106    37     Education         0.000    9.000    20.000
                         Iron              0.000   13.077    94.003
                         Folate            0.000  229.168  1586.876
                         Expenditures      0.000    8.817    14.752
                         Vitamin A         0.000  232.436  2982.945
                         Vitamin B12       0.000    2.111    31.241
                         Zinc              0.000    6.942   101.277
Nga MICS 37899    37     Education         0.000    0.450     1.000
                         Space pp          0.062    0.444    74.000
                         Wealth Score     -2.756   -0.221     2.941
Yem mVAM 54707    22     FCS               0.500   34.000   112.000
                         Expenditures pp   2.061    9.093    15.884
                         rCSI              0.000   14.000    56.000
Zwe MICS 10498    10     Education         0.000    0.450     1.000
                         Space pp          0.091    0.500     5.000
                         Wealth Score     -1.292   -0.455     3.219